In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI
_ = load_dotenv(find_dotenv())

zp_api_key = os.environ['ZHIPU_API_KEY']
llm = ChatOpenAI(base_url = "https://open.bigmodel.cn/api/paas/v4",
                 api_key = zp_api_key,
                 model = "glm-5.3",
                 temperature=0)



AIMessage(content='你好！我是一个由Z.ai训练的大型语言模型，你可以叫我GLM。\n\n我的主要工作是理解和生成文本，比如回答问题、提供信息、进行对话、创作内容等等。我通过学习海量的文本数据来获得这些能力，并致力于成为一个有用、无害的AI助手。\n\n有什么可以帮您的吗？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 975, 'prompt_tokens': 17, 'total_tokens': 992, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 907, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'glm-5.3', 'system_fingerprint': None, 'id': '2026091713352362960193dea84df4', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0addc-cc5d-7213-8f66-d911320a154b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 17, 'output_tokens': 975, 'total_tokens': 992, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 907}})

In [5]:
import sys
sys.path.append("Embedding.ipynb")
from zhipuai_embedding import ZhipuAIEmbeddings
from langchain_chroma import Chroma

embedding = ZhipuAIEmbeddings()
persist_directory = "data_base/vector_db"
vectordb = Chroma(persist_directory=persist_directory,  # 允许我们将persist_directory目录保存到磁盘上
                  embedding_function=embedding)
print(f"向量库中存储的数量：{vectordb._collection.count()}")

向量库中存储的数量：2939


In [12]:
# 测试
question = "窗口函数"
retriever = vectordb.as_retriever(search_kwargs={"k":3})
docs = retriever.invoke(question)
print(f"检索到的内容数:{len(docs)}")
for i, doc in enumerate(docs):
    print(f'检索到的第 {i+1} 个内容为：\n{doc.page_content}\n{"-"*50}\n')

检索到的内容数:3
检索到的第 1 个内容为：
APPENDIX A Window Function Refresher The recipes in this book take full advantage of the window functions added to the ISO SQL standard in 2003, as well as vendor-specific window functions. This appen‐ dix is meant to serve as a brief overview of how window functions work. Window functions make many typically difficult tasks (difficult to solve using standard SQL, that is) quite easy. For a complete list of window functions available, full syntax, and in-depth coverage of how they work, please
--------------------------------------------------

检索到的第 2 个内容为：
各函数详细说明

函数 语法说明 主要用途 LAG(expr, offset, default) 返回当前行之前第 offset 行（向上偏移）的 expr 值。默认 offset 为 1， default 为 NULL。 环比计算、查询上一周期数据。 LEAD(expr, offset, default) 返回当前行之后第 offset 行（向下偏移）的 expr 值，参数意义同 LAG 。 后续趋势判断、查询下一周期数据。 FIRST_VALUE(expr) 返回分区内第一行的 expr 值，需要配合窗口帧子句控制“第一行”的起始位置。 计算组内初始值、标记基线数据。 LAST_VALUE(expr) 返回分区内最后一行的 expr 值，但默认窗口帧会限制其仅能看到当前行，需指定完整窗口范围。 计算组内最终值、最新状态读取。

语法与窗口帧注意事项
-----------------------------

In [14]:
# 检索链

from langchain_core.runnables import RunnableLambda
def combine_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

combiner = RunnableLambda(combine_docs)
retrieval_chain = retriever | combiner

retrieval_chain.invoke("窗口函数")

'APPENDIX A Window Function Refresher The recipes in this book take full advantage of the window functions added to the ISO SQL standard in 2003, as well as vendor-specific window functions. This appen‐ dix is meant to serve as a brief overview of how window functions work. Window functions make many typically difficult tasks (difficult to solve using standard SQL, that is) quite easy. For a complete list of window functions available, full syntax, and in-depth coverage of how they work, please\n\n各函数详细说明\n\n函数 语法说明 主要用途 LAG(expr, offset, default) 返回当前行之前第 offset 行（向上偏移）的 expr 值。默认 offset 为 1， default 为 NULL。 环比计算、查询上一周期数据。 LEAD(expr, offset, default) 返回当前行之后第 offset 行（向下偏移）的 expr 值，参数意义同 LAG 。 后续趋势判断、查询下一周期数据。 FIRST_VALUE(expr) 返回分区内第一行的 expr 值，需要配合窗口帧子句控制“第一行”的起始位置。 计算组内初始值、标记基线数据。 LAST_VALUE(expr) 返回分区内最后一行的 expr 值，但默认窗口帧会限制其仅能看到当前行，需指定完整窗口范围。 计算组内最终值、最新状态读取。\n\n语法与窗口帧注意事项\n\n窗口函数分类详解\n\n聚合窗口函数：SUM、AVG、COUNT、MIN、MAX\n\n聚合窗口函数在保留每一行原始数据的同时，基于一个可移动或固定的窗口范围进行聚合计算。与普通聚合函数（GROUP BY）不同，窗口

In [17]:
# 创建检索问答链

from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

template = """
使用以下上下文来回答最后的问题。如果你不知道答案，就说你不知道，不要试图编造答
案。最多使用三句话。尽量使答案简明扼要。请你在回答的最后说“谢谢你的提问！”。

上下文：{context}

问题: {input}
"""

prompt = PromptTemplate(template=template)


qa_chain = (
    RunnableParallel({"context": retrieval_chain, "input": RunnablePassthrough()})
    | prompt
    | llm
    | StrOutputParser()
    )

question_1 = "什么是窗口函数？"
question_2 = "SQL必知必会是谁写的？"

result = qa_chain.invoke(question_1)
print("llm + 知识库：")
print(result)
result = qa_chain.invoke(question_2)
print("llm + 知识库：")
print(result)
result = llm.invoke(question_1).content
print("llm：")
print(result)
result = llm.invoke(question_2).content
print("llm：")
print(result)

llm + 知识库：
窗口函数是ISO SQL标准于2003年引入的SQL函数，它在一个定义的行集合（即“窗口”）上执行聚合计算，同时保留结果集中每一行的原始数据。与普通GROUP BY聚合函数不同，窗口函数不会将多行合并为一个输出值，而是为每行附加一个聚合值，常用于累计求和、移动平均等分析任务。DB2称之为OLAP函数，Oracle称之为分析函数，但ISO SQL标准称之为窗口函数。

谢谢你的提问！
llm + 知识库：
《SQL必知必会》的作者是美国作家 Ben Forta（本·福塔）。根据上下文，《MySQL 必知必会》正是他应众多读者请求，在《SQL必知必会》之后专门针对 MySQL 用户编写的。

谢谢你的提问！
llm：
# 窗口函数

窗口函数是 SQL 中一类特殊的函数，它能**在不减少行数的前提下**，对一组相关行进行计算。它相当于为每一行数据开一个“窗口”（可以看见上下相关的行），在窗口内进行计算，再把结果附加到当前行上。

## 与 GROUP BY 的核心区别

| | GROUP BY 聚合 | 窗口函数 |
|---|---|---|
| 返回行数 | 每组只返回 1 行 | 保留所有原始行 |
| 计算结果 | 替换原有数据 | 作为新列附加 |

## 基本语法

```sql
函数名(参数) OVER (
    PARTITION BY 分组字段    -- 定义分区（窗口）
    ORDER BY 排序字段        -- 窗口内排序
    ROWS BETWEEN ... AND ... -- 可选：帧（frame）范围
)
```

## 示例对比

假设有员工表：

| name | dept | salary |
|------|------|--------|
| Alice | 技术 | 10000 |
| Bob | 技术 | 8000 |
| Carol | 销售 | 6000 |

**用 GROUP BY**（只剩 2 行）：

```sql
SELECT dept, AVG(salary) AS avg_sal
FROM emp GROUP BY dept;
```

**用窗口函数**（每行都在，且带上部门均值）：

```sql
SELECT name, dept, salary,
     

In [ ]:
# 向检索链添加聊天记录

from langchain_core.prompts import ChatPromptTemplate

# 问答链的系统prompt
system_prompt = (
    "你是一个问答任务的助手。 "
    "请使用检索到的上下文片段回答这个问题。 "
    "如果你不知道答案就说不知道。 "
    "请使用简洁的话语回答用户。"
    "\n\n"
    "{context}"
)
# 制定prompt template
qa_prompt = ChatPromptTemplate(
    [
        ("system", system_prompt),
        ("placeholder", "{chat_history}"),
        ("human", "{input}"),
    ]
)
     

# 无历史记录
messages = qa_prompt.invoke(
    {
        "input": "窗口函数是什么？",
        "chat_history": [],
        "context": ""
    }
)
for message in messages.messages:
    print(message.content)


你是一个问答任务的助手。 请使用检索到的上下文片段回答这个问题。 如果你不知道答案就说不知道。 请使用简洁的话语回答用户。


窗口函数是什么？


In [20]:
# 有历史记录
messages = qa_prompt.invoke(
    {
        "input": "你可以介绍一下他吗？",
        "chat_history": [
            ("human", "窗口函数是什么？"),
            ("ai", "SQL 中一类特殊的函数，它能**在不减少行数的前提下**，对一组相关行进行计算。它相当于为每一行数据开一个“窗口”（可以看见上下相关的行），在窗口内进行计算，再把结果附加到当前行上。"),
        ],
        "context": ""
    }
)
for message in messages.messages:
    print(message.content)

你是一个问答任务的助手。 请使用检索到的上下文片段回答这个问题。 如果你不知道答案就说不知道。 请使用简洁的话语回答用户。


窗口函数是什么？
SQL 中一类特殊的函数，它能**在不减少行数的前提下**，对一组相关行进行计算。它相当于为每一行数据开一个“窗口”（可以看见上下相关的行），在窗口内进行计算，再把结果附加到当前行上。
你可以介绍一下他吗？


```python
RunnableBranch(
    (条件A, 执行动作A),   # <--- 相当于 if 条件A: 执行A
    (条件B, 执行动作B),   # <--- 相当于 elif 条件B: 执行B
    默认执行动作          # <--- 相当于 else: 默认执行
)

In [21]:
# 带有信息压缩的检索链

from langchain_core.runnables import RunnableBranch

# 压缩问题的系统 prompt
condense_question_system_template = (
    "请根据聊天记录完善用户最新的问题，"
    "如果用户最新的问题不需要完善则返回用户的问题。"
    )
# 构造 压缩问题的 prompt template
condense_question_prompt = ChatPromptTemplate([
        ("system", condense_question_system_template),
        ("placeholder", "{chat_history}"),
        ("human", "{input}"),
    ])
# 构造检索文档的链
# RunnableBranch 会根据条件选择要运行的分支
retrieve_docs = RunnableBranch(
    # 分支 1: 若聊天记录中没有 chat_history 则直接使用用户问题查询向量数据库
    (lambda x: not x.get("chat_history", False), (lambda x: x["input"]) | retriever, ),
    # 分支 2 : 若聊天记录中有 chat_history 则先让 llm 根据聊天记录完善问题再查询向量数据库
    condense_question_prompt | llm | StrOutputParser() | retriever,
)

In [22]:
# 重新定义 combine_docs
def combine_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs["context"]) # 将 docs 改为 docs["context"]
# 定义问答链
qa_chain = (
    RunnablePassthrough.assign(context=combine_docs) # 使用 combine_docs 函数整合 qa_prompt 中的 context
    | qa_prompt # 问答模板
    | llm
    | StrOutputParser() # 规定输出的格式为 str
)
# 定义带有历史记录的问答链
qa_history_chain = RunnablePassthrough.assign(
    context = (lambda x: x) | retrieve_docs # 将查询结果存为 content
    ).assign(answer=qa_chain) # 将最终结果存为 answer

In [23]:
# 不带聊天记录
qa_history_chain.invoke({
    "input": "窗口函数是什么?",
    "chat_history": []
})

{'input': '窗口函数是什么?',
 'chat_history': [],
 'context': [Document(id='22ac5f87-9126-4dd3-ad71-25770b29afd1', metadata={'trapped': '', 'author': 'Anthony Molinaro and Robert de Graaf', 'title': 'SQL Cookbook', 'creationdate': '2020-11-03T15:19:13+00:00', 'subject': '', 'file_path': 'data_base\\SQL Cookbook Query Solutions and Techniques for All SQL Users (Anthony Molinaro, Robert de Graaf) (z-library.sk, 1lib.sk, z-lib.sk).pdf', 'modDate': "D:20201103144248-05'00'", 'total_pages': 572, 'keywords': '', 'format': 'PDF 1.6', 'creator': 'AH CSS Formatter V6.2 MR4 for Linux64 : 6.2.6.18551 (2014/09/24 15:00JST)', 'source': 'data_base\\SQL Cookbook Query Solutions and Techniques for All SQL Users (Anthony Molinaro, Robert de Graaf) (z-library.sk, 1lib.sk, z-lib.sk).pdf', 'creationDate': 'D:20201103151913Z', 'page': 529, 'moddate': '2020-11-03T14:42:48-05:00', 'producer': 'Antenna House PDF Output Library 6.2.609 (Linux64)'}, page_content='APPENDIX A Window Function Refresher The recipes in thi

In [24]:
# 带聊天记录
qa_history_chain.invoke({
    "input": "OVER()怎么使用",
    "chat_history": [
        ("human", "窗口函数是什么?"),
        ("ai", "SQL 中一类特殊的函数，它能**在不减少行数的前提下**，对一组相关行进行计算。它相当于为每一行数据开一个“窗口”（可以看见上下相关的行），在窗口内进行计算，再把结果附加到当前行上。"),
    ]
})

{'input': 'OVER()怎么使用',
 'chat_history': [('human', '窗口函数是什么?'),
  ('ai',
   'SQL 中一类特殊的函数，它能**在不减少行数的前提下**，对一组相关行进行计算。它相当于为每一行数据开一个“窗口”（可以看见上下相关的行），在窗口内进行计算，再把结果附加到当前行上。')],
 'context': [Document(id='5b8d9835-303d-4f18-b931-6af23bfb6ff5', metadata={'source': 'data_base\\SQL 窗口函数教程.md'}, page_content='窗口函数在每一行上执行计算，并保留明细数据。例如：\n\n-- 为每一行都附加其所属部门的平均薪资，行数不变\nSELECT\n    employee_id,\n    department_id,\n    salary,\n    AVG(salary) OVER (PARTITION BY department_id) AS dept_avg_salary\nFROM employees;\n\n另外，普通聚合函数中 SUM() 等不会逐行累加，除非借助自连接或子查询；而窗口函数天然支持逐行滚动计算。执行顺序上，窗口函数在 SQL 逻辑查询的“结果集确定后”才执行（在 ORDER BY 之前），因此不能直接在 WHERE 或 HAVING 子句中使用窗口函数进行过滤，通常需要借助子查询或 CTE。\n\n窗口函数的基本语法结构\n\n窗口函数的标准语法如下：\n\n窗口函数名([表达式]) OVER (\n    [PARTITION BY 分区列]\n    [ORDER BY 排序列 [ASC | DESC]]\n    [ROWS | RANGE 窗口边界]\n)\n\n各部分含义：'),
  Document(id='8ac48058-33bd-4c3f-9f1f-95dd788d3d09', metadata={'source': 'data_base\\SQL 窗口函数教程.md'}, page_content='各部分含义：\n\n窗口函数名：如 ROW_NUMBER(), RANK(), DENSE_RANK(), SUM(), AVG(), LA